# Cont.IA — Treino do Modelo YOLO Próprio

**Objetivo:** Treinar um modelo YOLO11 com o dataset de correções coletadas pelo app,
eliminando a dependência da licença comercial da Ultralytics.

**Pré-requisitos:**
1. Executar `scripts/export_dataset.py` localmente para exportar o dataset
2. Fazer upload da pasta `Dataset_correcoes/` para o Google Drive
3. Executar este notebook no Google Colab com GPU (Runtime → Change runtime type → T4 GPU)

---

## 1. Conectar ao Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive montado')

## 2. Verificar GPU disponível

In [ ]:
import torch
print(f'GPU disponível: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Nenhuma — use Runtime > Change runtime type > T4 GPU"}')

## 3. Instalar Ultralytics

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO
print('✅ Ultralytics instalado')

## 4. Configurar caminhos do dataset

In [ ]:
import os
from pathlib import Path

# ⚙️ AJUSTE: caminho do dataset no seu Google Drive
DATASET_PATH = '/content/drive/MyDrive/contia/Dataset_correcoes'
DATA_YAML    = f'{DATASET_PATH}/data.yaml'

# Verifica se o dataset existe
assert Path(DATA_YAML).exists(), (
    f'❌ data.yaml não encontrado em {DATA_YAML}\n'
    'Certifique-se de ter executado scripts/export_dataset.py '
    'e feito upload para o Drive.'
)

# Conta imagens
train_imgs = list(Path(f'{DATASET_PATH}/images/train').glob('*.jpg'))
val_imgs   = list(Path(f'{DATASET_PATH}/images/val').glob('*.jpg'))
print(f'✅ Dataset encontrado')
print(f'   Treino     : {len(train_imgs)} imagens')
print(f'   Validação  : {len(val_imgs)} imagens')

# Exibe as classes
import yaml
with open(DATA_YAML) as f:
    data = yaml.safe_load(f)
print(f'   Classes ({data["nc"]}): {list(data["names"].values())}')

## 5. Treinar o modelo

Partimos do **YOLO11n** (nano — mais leve) já pré-treinado no COCO.
O fine-tuning com os dados de correção especializa o modelo para
os objetos específicos do app (garrafas, itens de mercado, etc.).

| Parâmetro | Valor | Por que |
|-----------|-------|--------|
| `model` | yolo11n.pt | Mais leve, roda bem em CPU no servidor |
| `epochs` | 100 | Suficiente para fine-tuning com dataset pequeno |
| `imgsz` | 640 | Padrão YOLO — compatível com o backend atual |
| `batch` | 16 | Adequado para T4 (16 GB VRAM) |
| `patience` | 20 | Para o treino se não melhorar em 20 épocas |


In [ ]:
model = YOLO('yolo11n.pt')  # Baixa automaticamente do Ultralytics

results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=0 if torch.cuda.is_available() else 'cpu',
    project='/content/drive/MyDrive/contia/treino',
    name='contia_v1',
    exist_ok=True,
    verbose=True,
)

print('\n✅ Treino concluído!')
print(f'   mAP50   : {results.results_dict.get("metrics/mAP50(B)", 0):.3f}')
print(f'   mAP50-95: {results.results_dict.get("metrics/mAP50-95(B)", 0):.3f}')

## 6. Avaliar o modelo treinado

In [ ]:
best_model_path = '/content/drive/MyDrive/contia/treino/contia_v1/weights/best.pt'
model_best = YOLO(best_model_path)

metrics = model_best.val(data=DATA_YAML)
print(f'\n📊 Métricas de validação:')
print(f'   Precisão  : {metrics.results_dict.get("metrics/precision(B)", 0):.3f}')
print(f'   Recall    : {metrics.results_dict.get("metrics/recall(B)", 0):.3f}')
print(f'   mAP50     : {metrics.results_dict.get("metrics/mAP50(B)", 0):.3f}')
print(f'   mAP50-95  : {metrics.results_dict.get("metrics/mAP50-95(B)", 0):.3f}')

## 7. Exportar e substituir no servidor

Após treinar, substitua o modelo no servidor:
1. Baixe o arquivo `best.pt` do Drive
2. Renomeie para `contia_v1.pt`
3. Coloque em `backend/contia_v1.pt`
4. Atualize `backend/.env`: `YOLO_MODEL=contia_v1.pt`
5. Restart: `docker-compose restart backend`


In [ ]:
print('📁 Modelo salvo em:')
print(f'   {best_model_path}')
print('\n⬇️  Para baixar diretamente do Colab:')
from google.colab import files
files.download(best_model_path)

## 8. Interpretação dos resultados

| mAP50 | Qualidade | O que fazer |
|-------|-----------|-------------|
| < 0.30 | Insuficiente | Coletar mais imagens e correções |
| 0.30–0.50 | Razoável | Usar em produção com limiar conf=0.35+ |
| 0.50–0.70 | Bom | Produção com limiar padrão (conf=0.25) |
| > 0.70 | Excelente | Produção — melhor que o YOLO genérico |

**Dica:** Se o resultado for insuficiente, colete mais correções pelo app
e execute novamente este notebook em 2–4 semanas.